# Ôn tập Buổi 04 - Data Loading

        **Thời lượng gợi ý:** 45-60 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Dùng `Path` và đọc CSV có header/separator/missing khác nhau.
- Kiểm tra dữ liệu ngay sau khi load.
- Đọc file lớn theo chunk mà không nạp toàn bộ vào RAM.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi4_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import json
import pandas as pd
from io import StringIO
DATA_DIR = ROOT / 'datasets' / 'pydata_book_examples'
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. File không có header thì tham số nào cần dùng?**

<details><summary>Kiểm tra đáp án</summary>

`header=None`, thường đi cùng `names=[...]`.

</details>

**2. `na_values` có tác dụng gì?**

<details><summary>Kiểm tra đáp án</summary>

Khai báo thêm các sentinel trong file cần coi là missing.

</details>

**3. `chunksize=1000` trả về DataFrame hay iterator?**

<details><summary>Kiểm tra đáp án</summary>

Một `TextFileReader` để lặp qua từng DataFrame nhỏ.

</details>


## 1. Quy trình Load -> Inspect -> Validate


In [ ]:
ex1 = pd.read_csv(DATA_DIR / "ex1.csv")
display(ex1.head())
print(ex1.shape)
display(ex1.dtypes.rename("dtype"))
print("missing:", ex1.isna().sum().sum())
assert ex1.shape == (3, 5)


## 2. Không có header và separator là khoảng trắng


In [ ]:
names = ["a", "b", "c", "d", "message"]
ex2 = pd.read_csv(DATA_DIR / "ex2.csv", header=None, names=names)
ex3 = pd.read_csv(DATA_DIR / "ex3.txt", sep=r"\s+")
display(ex2)
display(ex3.head())
assert ex2.columns.tolist() == names
assert ex3.shape[1] == 3


## 3. Bỏ dòng chú thích và khai báo missing


In [ ]:
ex4 = pd.read_csv(DATA_DIR / "ex4.csv", skiprows=[0, 2, 3])
ex5_default = pd.read_csv(DATA_DIR / "ex5.csv")
ex5_custom = pd.read_csv(
    DATA_DIR / "ex5.csv",
    keep_default_na=False,
    na_values={"message": ["NA"], "c": [""]},
)
display(ex4)
display(ex5_custom)
assert ex4.shape == (3, 5)
assert ex5_default.isna().sum().sum() == 2
assert ex5_custom.isna().sum().sum() == 2


## 4. File lớn: xử lý theo chunk

Ví dụ tính tổng cột `one` mà mỗi lần chỉ giữ một phần dữ liệu trong bộ nhớ.


In [ ]:
total = 0.0
row_count = 0
for chunk in pd.read_csv(DATA_DIR / "ex6.csv", chunksize=1000):
    total += chunk["one"].sum()
    row_count += len(chunk)
print("rows:", row_count, "| sum(one):", round(total, 4))
assert row_count > 1000


## 5. JSON: cấu trúc lồng nhau cần normalize


In [ ]:
raw = '''[
  {"id": 1, "student": {"name": "An", "class": "A"}, "scores": [8, 9]},
  {"id": 2, "student": {"name": "Binh", "class": "B"}, "scores": [7, 8]}
]'''
records = json.loads(raw)
flat = pd.json_normalize(records)
display(flat)
assert "student.name" in flat.columns


## Bài tự luyện

        Đọc `ex5.csv` với cột `something` làm index; coi chuỗi `NA` là missing nhưng giữ các quy ước missing mặc định. Kiểm tra shape và số missing theo cột.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        answer = pd.read_csv(DATA_DIR / "ex5.csv", index_col="something", na_values=["NA"])
print(answer.shape)
print(answer.isna().sum())
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi biết chọn `header`, `names`, `sep`, `index_col`, `skiprows`, `na_values`.
- [ ] Tôi kiểm tra shape/dtype/missing ngay sau khi đọc.
- [ ] Tôi biết khi nào dùng `nrows` và `chunksize`.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
